## Imports and Setup

In [50]:
import glob

import yaml
import h5py
import numpy as np

# load config file
yaml_file = open("config.yaml", 'r')
config = yaml.load(yaml_file, Loader=yaml.Loader)

## Create Train/Test/Val Virtual Datasets

In [126]:
# get .h5 files in data directory
files = glob.glob(config['dataset']['data_directory'] + '/*.h5')

# we assume we have these entries
assert(all(x in config['dataset']['entry_keys'] for x in ['data', 'T', 'fs']))

# create virtual sources for each entry key
sources_dict = {k : [] for k in config['dataset']['entry_keys']}
sh_dict = {}
total_length = 0
for i, filename in enumerate(files, start=1):
    with h5py.File(filename, 'r') as f:
        for k in config['dataset']['entry_keys']:
            
            # save entry data shapes
            if i == 1:
                sh_dict[k] = f[k].shape[1:]
            
            # only add one source for 'T' and 'fs' because they're assumed
            # to all be the same
            if (k == 'T' or k == 'fs') and i > 1:
                continue
            else:
                vsource = h5py.VirtualSource(f[k])
                sources_dict[k].append(vsource)
        
        # add to total length
        total_length += f['data'].shape[0]

# make layouts
layout_dict = {}
for k in config['dataset']['entry_keys']:
    if k != 'T' and k!= 'fs':
        layout_dict[k] = h5py.VirtualLayout(shape=(total_length,)+sh_dict[k], dtype=np.float64)
    else:
        layout_dict[k] = h5py.VirtualLayout(shape=(1,)+sh_dict[k], dtype=np.float64)

# fill layouts
for k in config['dataset']['entry_keys']:
    offset = 0
    for vsource in sources_dict[k]:
        length = vsource.shape[0]
        layout_dict[k][offset:offset+length] = vsource
        offset += length
        
# create virtual datasets
with h5py.File(config['dataset']['data_directory'] + "/VSD_main.h5", 'w', libver='latest') as f:
    for k in config['dataset']['entry_keys']:
        f.create_virtual_dataset(k, layout_dict[k], fillvalue=0)